In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [1]:
import pandas as pd
import numpy as np

In [2]:

def read_file(fname, unescape_newline=True):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    if unescape_newline:
      output = [l.replace("\\n", "\n") for l in output]
    return output

def get_candidates(df, lp, split="dev", n=100, model_name="TowerInstruct-7B-w-chat-empty-sys", context_type="full_context_empty_sys", data_name="wmt24_chat"):
  source_lang = lp.split("-")[0]
  target_lang = lp.split("-")[1].replace('pt','pt-br')
  try:
    xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
    yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split.replace('test', 'test_blind')}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")
  except:
    xx_yy_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{source_lang}-{target_lang}/100_epsilon_candidates.txt")
    yy_xx_generations = read_file(f"../candidates/{context_type}/{model_name}/{data_name}_{split}.{target_lang}-{source_lang}/100_epsilon_candidates.txt")


  xx_yy_generations = np.array(xx_yy_generations).reshape((len(xx_yy_generations)//n, n))
  yy_xx_generations = np.array(yy_xx_generations).reshape((len(yy_xx_generations)//n, n))

  assert len(xx_yy_generations) + len(yy_xx_generations) == len(df)
  translations = []
  j,k=0,0

  for i, row in df.iterrows():
    if row["lp"] == f"{source_lang}-{target_lang}":
      translations.append(xx_yy_generations[j])
      j+=1
    else:
      # nl-en and it-en are considered here
      translations.append(yy_xx_generations[k])
      k+=1
  return np.array(translations)

In [3]:
def get_dataframe(split, lang_pair, data_name='wmt24_chat', 
                  model_name='TowerInstruct-7B-w-chat-empty-sys', 
                  context_type="full_context_empty_sys"):
    df = pd.read_csv(f"../tacl_results/{data_name}/{split}.{lang_pair}.csv",
                    keep_default_na=False,
        na_values=["NaN"],)
    df["lp"] = df["source_language"] + "-" + df["target_language"]

    output_reshaped = get_candidates(df, lang_pair, split, data_name=data_name, model_name=model_name, context_type=context_type)

    return df, output_reshaped

# check overlap between hypothesis

In [19]:
def get_overlap(df, output_reshaped_context, output_reshaped_no_context):
    count_overlap  = []
    for i in range(len(df)):
        intersect = len(set(output_reshaped_context[i]).intersection(set(output_reshaped_no_context[i])))
        union = len(set(output_reshaped_context[i]).union(set(output_reshaped_no_context[i])))
        count_overlap.append(intersect/union)
    return count_overlap

In [21]:
for lp in ["en-de", "en-fr", "en-ko", "en-nl", "en-pt"]:
    df, output_reshaped_context = get_dataframe("test", lp)
    _, output_reshaped_no_context = get_dataframe("test", lp, 
                                                model_name="TowerInstruct-7B-v0.2",
                                                context_type="no_context")
    count_overlap = get_overlap(df, output_reshaped_context, output_reshaped_no_context)
    print(lp, np.mean(count_overlap))

en-de 0.19088574172229775
en-fr 0.18085086824739988
en-ko 0.15383590679337542
en-nl 0.1725062734418849
en-pt 0.1878417803191727


In [ ]:
def get_exact_match(df, outputs):
    matches = []
    for i, row in df.iterrows():
        matches.append(row["reference"] in outputs[i])
    return matches

In [24]:
for lp in ["en-de", "en-fr", "en-ko", "en-nl", "en-pt"]:
    df, output_reshaped_context = get_dataframe("test", lp)
    _, output_reshaped_no_context = get_dataframe("test", lp, 
                                                model_name="TowerInstruct-7B-v0.2",
                                                context_type="no_context")
    ref_in_context = get_exact_match(df, output_reshaped_context)
    ref_in_no_context = get_exact_match(df, output_reshaped_no_context)
    print(lp, np.mean(ref_in_context), np.mean(ref_in_no_context))

en-de 0.5198432141107301 0.512983831455169
en-fr 0.5700621712099474 0.5351506456241033
en-ko 0.5847628657921292 0.38092835519677093
en-nl 0.5950372208436725 0.43970223325062036
en-pt 0.6387254901960784 0.40490196078431373


# Chrf oracle

In [5]:
import sacrebleu

def oracle_chrf(df, outputs):
    scores = []
    for i, row in df.iterrows():
        scores.append(np.max([sacrebleu.sentence_chrf(x, [row["reference"]]).score for x in outputs[i]]))
    return scores

for lp in ["en-de", "en-fr", "en-ko", "en-nl", "en-pt"]:
    df, output_reshaped_context = get_dataframe("test", lp)
    _, output_reshaped_no_context = get_dataframe("test", lp, 
                                                model_name="TowerInstruct-7B-v0.2",
                                                context_type="no_context")
    df["context-oracle"] = oracle_chrf(df, output_reshaped_context)
    df["no-context-oracle"] = oracle_chrf(df, output_reshaped_no_context)
    for lp, lp_df in df.groupby("lp"):
        print(lp, np.mean(lp_df["context-oracle"]), np.mean(lp_df["no-context-oracle"]))

de-en 91.77379722339437 92.91595256775533
en-de 90.2147975267435 88.91701319221163
en-fr 90.65064443830842 89.88952948873924
fr-en 93.19286568964348 94.80694102080092
en-ko 83.18376654414487 67.14945831849856
ko-en 93.5573805567849 92.14842091377571
en-nl 91.8200019297532 83.68758161381272
nl-en 91.92851805026001 91.24574002709969
en-pt-br 94.03989516585995 87.03278359762547
pt-br-en 91.993529427749 86.30350912238612


# Muda

In [25]:
from comet import download_model, load_from_checkpoint
comet_metric = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 501.26it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.2.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../home/sweta/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/371e9839ca4e213dde891b066cf3080f75ec7e72/checkpoints/model.ckpt`
Encoder model frozen.
